<!-- config-banner -->
# DINO 1° adjoint — poster panels (SciDAC PI meeting, Aug 2026)

**`visc2x`** — 5 yr adjoint from the 180 yr pickup, `viscAhD` = `viscAhZ` = 2× reference via `PARM05`. Run 28486.


## Loading the Data

In [1]:
import os
import numpy as np
import xarray as xr
import xmitgcm
import matplotlib.pyplot as plt

import warnings; warnings.filterwarnings("ignore")

In [2]:
## analyze a sample run directory (here 10 years adjoint)

run_dir = (
    "/scratch2/tshahriar/DINO_1deg_tapAdj_runs/"
    "DINO_1deg_tapAdj_5yr_from180yrPk_visc2x_run28486/"
)
run_dir

'/scratch2/tshahriar/DINO_1deg_tapAdj_runs/DINO_1deg_tapAdj_5yr_from180yrPk_visc2x_run28486/'

In [3]:
# Metadata for the three adjoint controls used by the poster.
# 'dims' are the MITgcm MDS dimension names that xmitgcm expects:
#   ('k','j','i')  -> 3D on tracer points   (Z, YC, XC)
#   ('j','i_g')    -> 2D on U      points   (YC, XG)   <- ADJtaux
adj_vars = {
    # --- 3D adjoint variables -----------------------------------------
    'ADJtheta': {
        'dims': ['k', 'j', 'i'],
        'attrs': {
            'standard_name': 'ADJtheta',
            'long_name': 'Adjoint Sensitivity to Potential Temperature',
            'units': 'dJ/degC'
        }
    },

    'ADJdiffkr': {
        'dims': ['k', 'j', 'i'],
        'attrs': {
            'standard_name': 'ADJdiffkr',
            'long_name': 'Adjoint Sensitivity to Vertical Diffusivity',
            'units': 'dJ/(m^2/s)'
        }
    },

    # --- 2D surface momentum adjoint ----------------------------------
    'ADJtaux': {
        'dims': ['j', 'i_g'],
        'attrs': {
            'standard_name': 'ADJtaux',
            'long_name': 'Adjoint Sensitivity to Zonal Surface Wind Stress',
            'units': 'dJ/(N/m^2)'
        }
    },
}

In [4]:
# Only the three controls the poster needs.  Reading fewer prefixes keeps
# open_mdsdataset fast; add names back here (and to adj_vars above) if you
# want ADJsalt, ADJtauy, ADJqnet, ADJqsw or ADJempmr.
adj_prefix = ['ADJtheta', 'ADJdiffkr', 'ADJtaux']

In [5]:
ds_adj = xmitgcm.open_mdsdataset(
    run_dir,
    grid_dir=run_dir,
    prefix=adj_prefix,
    ref_date=np.datetime64("2000-01-01T00:00:00"),
    delta_t=1800,
    extra_variables=adj_vars
)

In [6]:
ds_adj

<xarray.Dataset>
Dimensions:    (time: 366, Z: 36, YC: 198, XC: 51, XG: 51, YG: 198, Zp1: 37,
                Zu: 36, Zl: 36)
Coordinates: (12/37)
    iter       (time) int64 dask.array<chunksize=(1,), meta=np.ndarray>
  * time       (time) datetime64[ns] 2180-05-16 2180-05-21 ... 2185-05-15
  * XC         (XC) >f4 -50.5 -49.5 -48.5 -47.5 -46.5 ... -3.5 -2.5 -1.5 -0.5
  * YC         (YC) >f4 -69.85 -69.5 -69.15 -68.79 ... 68.43 68.79 69.15 69.5
  * XG         (XG) >f4 -50.0 -49.0 -48.0 -47.0 -46.0 ... -3.0 -2.0 -1.0 0.0
  * YG         (YG) >f4 -69.68 -69.33 -68.97 -68.61 ... 68.61 68.97 69.33 69.68
    ...         ...
    maskCtrlW  (Z, YC, XG) bool dask.array<chunksize=(36, 198, 51), meta=np.ndarray>
    maskCtrlS  (Z, YG, XC) bool dask.array<chunksize=(36, 198, 51), meta=np.ndarray>
    dxF        (YC, XC) >f4 dask.array<chunksize=(198, 51), meta=np.ndarray>
    dyF        (YC, XC) >f4 dask.array<chunksize=(198, 51), meta=np.ndarray>
    dxV        (YG, XG) >f4 dask.array<chunksize=(198, 51), meta=np.ndarray>
    dyU        (YG, XG) >f4 dask.array<chunksize=(198, 51), meta=np.ndarray>
Data variables:
    ADJdiffkr  (time, Z, YC, XC) float32 dask.array<chunksize=(1, 36, 198, 51), meta=np.ndarray>
    ADJtaux    (time, YC, XG) float32 dask.array<chunksize=(1, 198, 51), meta=np.ndarray>
    ADJtheta   (time, Z, YC, XC) float32 dask.array<chunksize=(1, 36, 198, 51), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    title:        netCDF wrapper of MITgcm MDS binary data
    source:       MITgcm
    history:      Created by calling `open_mdsdataset(grid_dir='/scratch2/tsh...

In [7]:
# --- quick checks (all cheap: no data is read off disk) ---------------

# vertical level used for the two 3D rows: index 14 ~ 231 m
print("Z[14] =", ds_adj.Z.values[14], "m")

# ADJtaux is 2D and sits on the U grid (YC, XG) -> needs the hFacW mask
print("ADJtheta ", ds_adj.ADJtheta.shape,  ds_adj.ADJtheta.dims)
print("ADJdiffkr", ds_adj.ADJdiffkr.shape, ds_adj.ADJdiffkr.dims)
print("ADJtaux  ", ds_adj.ADJtaux.shape,   ds_adj.ADJtaux.dims)

# adjoint time axis: 5-day output running backwards from the cost function
print("time:", ds_adj.time.values[0], "->", ds_adj.time.values[-1],
      f"({ds_adj.sizes['time']} steps)")

Z[14] = -230.986 m
ADJtheta  (366, 36, 198, 51) ('time', 'Z', 'YC', 'XC')
ADJdiffkr (366, 36, 198, 51) ('time', 'Z', 'YC', 'XC')
ADJtaux   (366, 198, 51) ('time', 'YC', 'XG')
time: 2180-05-16T00:00:00.000000000 -> 2185-05-15T00:00:00.000000000 (366 steps)


### Note on the reverse time order

I had to plot the ADJtheta file in the reverse order of the timestep. This is because, ami forward model er time step use kortesi ADJtheta file k numbering korar jonno. But adjoint runs backward in time. So jokhon plot korbo tokhon seta ke reverse timestep order e plot korte hobe.

"ADJtheta.0000000000.data" data in this file should closely corresponds to the data in the file "adxx_theta.0000000000.data"

This is why the export below uses `reverse_time=True`: saved frame `0000` is the **last** adjoint time (lead 0, right at the cost function) and the frame number counts *backwards* in time from there.

## Poster-ready frames for the 3 × 3 sensitivity grid

This notebook has been trimmed to just what the ImPACTS/SciDAC poster needs.
The `adxx` exploration, the `animate_ADJ_*` helpers and the per-variable
diagnostic loops are gone — they took a long time to run and produced figures
the poster never uses. The full version is kept in
`01_adj_field_animations.ipynb`.

The poster shows a 3 × 3 matrix: **rows** = control variable
(`ADJtheta`, `ADJdiffkr`, `ADJtaux`), **columns** = lead time before the cost
function (6 months, 1 year, final ≈ 5 years). The variable name is printed once
per row, rotated, to the left of the grid — so the panels themselves do not
need to repeat it, and the colorbar does not need a text label.

All nine panels come from **one call** to `save_poster_grid()`. That matters:
they only line up if every panel uses the same axes rectangle, and that
rectangle can only be worked out once all three rows' colour scales are known.

How the export works:

1. **Fixed output size.** The old `save_ADJ_3d_frames()` used
   `bbox_inches="tight"`, so the frame size depended on how wide the colorbar
   label happened to be — `ADJtheta` came out 838 × 468 while `ADJdiffkr` came
   out 860 × 468. Here the figure size is fixed, so every frame is exactly
   1408 × 640 px.
2. **Nothing clipped, no wasted margin.** `_fit_axes_in_figure()` measures the
   real text extents after drawing and rescales the axes so the drawn content
   exactly fills the frame — the tight-bbox result, without changing the frame
   size.
3. **Columns aligned.** `_colorbar_formatter()` forces every colorbar into
   scientific notation with a shared ×10ⁿ header, so all three rows have
   tick labels one or two characters wide instead of `−0.04`-style ones.
   `_common_layout()` then measures all three rows and adopts the tightest
   result for all nine panels, so the maps and colorbars sit in identical
   positions.
4. **Shared axes.** Latitude labels and ticks on the **left column** only,
   longitude on the **bottom row** only, exactly as a subplot grid would.
5. **A frame shape that fills the poster block** — 2.2 : 1, see the next cell.
6. **PNG, not JPEG.** These panels are line art: text, axis rules and
   hard-edged `pcolormesh` cells. JPEG's DCT rings around exactly those
   high-contrast edges. PNG is lossless and, because the maps are mostly flat
   colour, actually came out *smaller* — ~55 kB per panel.

Only the last code cell does real work: three probe figures for the layout plus
nine panels.

In [8]:
# ======================================================================
#  POSTER GEOMETRY  ->  frame aspect ratio
# ----------------------------------------------------------------------
#  The sensitivity block on the poster is 0.705 x \textwidth wide, and
#  \textwidth = 121.92 cm - 2 x 2.2 cm margins = 117.52 cm, so:
#
#      block inner width (minus tcolorbox padding)     ~ 81.9 cm
#      minus the rotated row label                     ~  1.2 cm
#      minus 3 column gutters (0.013 \linewidth each)  ~  3.2 cm
#      -------------------------------------------------------
#      => width of one panel                           ~ 25.6 cm
#
#      height left for the grid after the block title,
#      the column headings and the caption             ~ 35.0 cm
#      => height of one panel (3 rows)                 ~ 11.6 cm
#
#      => frame aspect = 25.6 / 11.6                   ~ 2.2
#
#  The old frames were 838/468 = 1.79, which is TALLER than the cell can
#  be, so LaTeX had to shrink them to fit the height and ~20% of the
#  block's width went unused.  2.2 fills it.
#
#  AFTER you regenerate all nine panels at this aspect ratio, make the
#  poster match with two one-line edits:
#
#      Poster/sections/08_sensitivity_matrix.tex
#          \setsenswidth{0.310\linewidth}           % was 0.255
#
#      Poster/config/poster-style.tex
#          \setlength{\sensimgh}{0.4545\sensimgw}   % was 0.5585 (= 1/2.2)
#
#  Nothing breaks if you forget: each poster cell is a fixed box and the
#  frame is scaled to fit inside it, so a mismatch only shows up as white
#  bars around the panel.
# ======================================================================

POSTER_ASPECT = 2.2          # frame width / frame height

POSTER_STYLE = dict(
    fig_w          = 8.8,    # inches; height = 8.8 / 2.2 = 4.0 exactly, so
    dpi            = 160,    # every frame is exactly 1408 x 640 px
    title_size     = 15,
    label_size     = 13,
    tick_size      = 12,
    cbar_tick_size = 11,
    pad            = 0.012,  # margin left around the drawn content
    # Starting rectangles only.  _fit_axes_in_figure() re-measures the real
    # text extents after drawing and rescales these so that nothing is
    # clipped AND no margin is wider than it has to be, so these numbers no
    # longer have to be guessed correctly by hand.
    ax_rect        = (0.095, 0.180, 0.770, 0.712),   # left, bottom, w, h
    cb_rect        = (0.888, 0.180, 0.016, 0.712),
)

# The panel prints ~25.6 cm wide but is drawn 8.8 in = 22.4 cm wide, so the
# font sizes above land at roughly 1.15x their nominal pt size on the poster
# (title ~17 pt, ticks ~14 pt).  Poster body text is 21 pt for comparison.
#
# If you change POSTER_ASPECT, keep fig_w / POSTER_ASPECT a round number so
# the output stays an exact pixel size.

In [9]:
# ======================================================================
#  Helpers: grid-aware masking, robust colour limits, one styled panel
# ======================================================================
from matplotlib.ticker import ScalarFormatter, MaxNLocator


def _adj_grid_info(ds, da, zlev):
    """Plot coordinates and the land mask that matches this variable's grid.

    ADJtheta / ADJsalt / ADJdiffkr live on tracer points (XC, YC) -> hFacC
    ADJtaux                        lives on U      points (XG, YC) -> hFacW
    ADJtauy                        lives on V      points (XC, YG) -> hFacS

    Picking the mask from the variable's own dims matters: the 2D fields
    are NOT on the tracer grid, so masking them with hFacC (or not masking
    them at all) leaves land points drawn as if they were data.
    """
    xcoord = "XG" if "XG" in da.dims else "XC"
    ycoord = "YG" if "YG" in da.dims else "YC"
    kmask  = 0 if zlev is None else zlev

    if xcoord == "XG" and "hFacW" in ds:
        mask = ds.hFacW.isel(Z=kmask) > 0
    elif ycoord == "YG" and "hFacS" in ds:
        mask = ds.hFacS.isel(Z=kmask) > 0
    elif "hFacC" in ds:
        mask = ds.hFacC.isel(Z=kmask) > 0
    else:
        mask = None

    return xcoord, ycoord, mask


def _robust_vmax(A, p):
    """Symmetric colour limit at the p-th percentile of |A|, outlier-safe."""
    finite = np.isfinite(A)
    if not finite.any():
        return 1.0
    v = np.percentile(np.abs(A[finite]), p)
    if not np.isfinite(v) or v == 0:
        v = np.nanmax(np.abs(A[finite]))
    return float(v) if (np.isfinite(v) and v != 0) else 1.0


def prepare_ADJ_field(var, ds, zlev=None, p=96, ylim=None):
    """Masked single-level field plus ONE colour limit for the whole series.

    Using one limit for every frame is what makes a row comparable across
    columns.  Pass zlev for 3D controls; leave it None for 2D controls.

    ylim : optional (south, north) in degrees, e.g. (-55, 70), to crop the
           far south where these fields are essentially zero in every frame.
    """
    if var not in ds:
        raise KeyError(f"{var} not found in dataset.")

    da = ds[var]

    if "Z" in da.dims:
        if zlev is None:
            raise ValueError(f"{var} is 3D - pass zlev=...")
        da = da.isel(Z=zlev)

    xcoord, ycoord, mask = _adj_grid_info(ds, da, zlev)
    if mask is not None:
        da = da.where(mask)

    if ylim is not None:
        da = da.sel({ycoord: slice(*ylim)})

    vmax = _robust_vmax(da.values, p)
    return da, xcoord, ycoord, vmax


def _colorbar_formatter():
    """Scientific notation with a shared x10^n header on every colorbar.

    This is the fix for the columns not lining up.  Left to itself
    matplotlib labelled the three rows very differently:

        ADJtheta    -2  -1  0  1  2          (with a 1e-5 header)
        ADJdiffkr   -0.04 -0.02 0.00 ...     (no header, MUCH wider)
        ADJtaux     -3  -2  -1  0  1  2  3   (with a 1e-7 header)

    The wide diffkr numbers ate ~4% of that panel's width, so its map came
    out narrower and shifted relative to the other two.  Forcing the power
    of ten out into a header makes every row's tick labels one or two
    characters, and as a bonus the rows become easier to compare.

    A fresh instance is returned each call: a Formatter binds to one axis.
    """
    fmt = ScalarFormatter(useMathText=True)
    fmt.set_powerlimits((0, 0))       # always factor out a power of ten
    return fmt


def _get_renderer(fig):
    """A renderer we can measure text with, whatever backend is active."""
    try:
        return fig.canvas.get_renderer()
    except AttributeError:
        from matplotlib.backends.backend_agg import FigureCanvasAgg
        FigureCanvasAgg(fig)
        return fig.canvas.get_renderer()


def _fit_axes_in_figure(fig, axes_list, pad=0.012, n_iter=3):
    """Fit the axes group into the figure without changing the figure size.

    This replaces bbox_inches="tight".  It measures everything that will
    actually be drawn -- tick labels, axis labels, the title, the colorbar
    header -- and then rescales/translates the axes rectangles so that this
    drawn extent exactly fills the figure minus `pad`.

    Two consequences, both wanted here: nothing is ever clipped, and no
    margin is larger than it needs to be.  The figure size is untouched, so
    every exported frame is still exactly the same number of pixels.
    """
    for _ in range(n_iter):
        fig.canvas.draw()
        r   = _get_renderer(fig)
        inv = fig.transFigure.inverted()

        boxes = [a.get_tightbbox(r).transformed(inv) for a in axes_list]
        x0 = min(b.x0 for b in boxes); x1 = max(b.x1 for b in boxes)
        y0 = min(b.y0 for b in boxes); y1 = max(b.y1 for b in boxes)

        if (x1 - x0) <= 0 or (y1 - y0) <= 0:
            break

        sx = (1 - 2 * pad) / (x1 - x0)
        sy = (1 - 2 * pad) / (y1 - y0)

        if abs(sx - 1) < 2e-3 and abs(sy - 1) < 2e-3:
            break

        for a in axes_list:
            p = a.get_position()
            a.set_position([
                pad + (p.x0 - x0) * sx,
                pad + (p.y0 - y0) * sy,
                p.width  * sx,
                p.height * sy,
            ])

    return fig


def _make_panel(da, i, xcoord, ycoord, vmax, title,
                show_x=True, show_y=True, add_colorbar=True,
                rects=None, style=None):
    """Build one panel at a fixed figure size.

    show_x / show_y : draw the longitude / latitude label and tick numbers.
        The poster grid shares axes, so only the LEFT column carries
        latitude and only the BOTTOM row carries longitude.

    rects : (ax_rect, cb_rect) to use verbatim.  Passing the same pair to
        all nine panels is what makes the maps and the colorbars line up
        exactly; leave it None to start from the POSTER_STYLE defaults.
    """
    st = {**POSTER_STYLE, **(style or {})}
    fig_h = st["fig_w"] / POSTER_ASPECT

    ax_rect, cb_rect = rects if rects is not None else (st["ax_rect"],
                                                        st["cb_rect"])

    fig = plt.figure(figsize=(st["fig_w"], fig_h), dpi=st["dpi"])
    ax  = fig.add_axes(ax_rect)

    pcm = da.isel(time=i).plot(
        x=xcoord, y=ycoord,
        cmap="RdBu_r",
        vmin=-vmax, vmax=vmax,
        add_colorbar=False,
        add_labels=False,      # we set our own, at poster sizes
        ax=ax,
    )

    ax.tick_params(labelsize=st["tick_size"])
    ax.set_title(title, fontsize=st["title_size"], pad=5)

    if show_y:
        ax.set_ylabel("Latitude [°N]", fontsize=st["label_size"])
    else:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)

    if show_x:
        ax.set_xlabel("Longitude [°E]", fontsize=st["label_size"])
    else:
        ax.set_xlabel("")
        ax.tick_params(labelbottom=False)

    cax = None
    if add_colorbar:
        cax  = fig.add_axes(cb_rect)
        cbar = fig.colorbar(
            pcm, cax=cax,
            format=_colorbar_formatter(),
            ticks=MaxNLocator(nbins=5, symmetric=True),
        )
        cbar.set_label("")     # <-- deliberately blank: the poster labels
                               #     each ROW instead of each colorbar
        cbar.ax.tick_params(labelsize=st["cbar_tick_size"])
        cbar.ax.yaxis.get_offset_text().set_fontsize(st["cbar_tick_size"])

    return fig, ax, cax


def _common_layout(fields, style=None,
                   probe_title="0000-00-00   •   depth = 000 m"):
    """One pair of axes rectangles that suits every row of the grid.

    Each row is drawn once with the FULL set of labels and auto-fitted;
    the tightest result -- the row whose colorbar numbers are widest, so
    whose map ends up narrowest -- is then adopted for all nine panels.
    Every panel therefore has its map and its colorbar in exactly the same
    place, which is what lines the columns up.

    A row with narrower labels simply ends up with a little more white
    space, rather than a differently sized map.
    """
    st = {**POSTER_STYLE, **(style or {})}
    best = None

    for (da, xcoord, ycoord, vmax) in fields:
        fig, ax, cax = _make_panel(da, 0, xcoord, ycoord, vmax, probe_title,
                                   show_x=True, show_y=True, style=style)
        _fit_axes_in_figure(fig, [ax, cax], pad=st["pad"])
        rects = (tuple(ax.get_position().bounds),
                 tuple(cax.get_position().bounds))
        plt.close(fig)

        if best is None or rects[0][2] < best[0][2]:   # smallest map width
            best = rects

    return best

In [10]:
# ======================================================================
#  Frame export
# ======================================================================

def poster_frame_numbers(ds, stride=4, leads_days=(180, 365, None), quiet=False):
    """Turn lead times (days before the cost function) into frame numbers.

    Frames are written in reverse time order, so frame 0 is the LAST adjoint
    time (lead 0) and each step back is stride * dt days.  `None` means the
    final frame of the sequence.

    With dt = 5 d and stride = 4 this gives 20 d per saved frame, i.e.
    6 months -> 9, 1 year -> 18, final -> 91.
    """
    t = ds.time.values
    dt_days = int(np.round((t[1] - t[0]) / np.timedelta64(1, "D")))
    step    = stride * dt_days
    nmax    = (len(t) - 1) // stride

    out = {lead: (nmax if lead is None else int(round(lead / step)))
           for lead in leads_days}

    if not quiet:
        print(f"dt = {dt_days} d, stride = {stride} -> {step} d per frame; "
              f"final frame = {nmax}")
        for lead, n in out.items():
            tag = "final" if lead is None else f"{lead} d"
            print(f"    {tag:>8s} -> frame {n:4d}")

    return out


# ----------------------------------------------------------------------
#  The poster grid: 3 rows x 3 columns, exported in ONE call so that the
#  layout can be shared across all nine panels.
# ----------------------------------------------------------------------

POSTER_ROWS = [
    dict(var="ADJtheta",  stem="sens_theta",  zlev=14),
    dict(var="ADJdiffkr", stem="sens_diffkr", zlev=14),
    dict(var="ADJtaux",   stem="sens_taux",   zlev=None),   # 2D, U grid
]


def save_poster_grid(
    ds, rows=None, outdir=os.path.join(run_dir, "figures", "poster_frames"), stride=4, p=96, ylim=None,
    filetype="png", style=None,
    leads=(180, 365, None), suffixes=("06mo", "01yr", "final"),
):
    """Export all nine poster panels as one aligned set.

    Why one call instead of three: the panels only line up if every one of
    them uses the SAME axes rectangle, and that rectangle can only be
    worked out once all three rows' colour scales (and hence their colorbar
    label widths) are known.  Exporting a single row on its own would give
    it a layout of its own and put it out of step with the other two.

    Axes are shared the way a subplot grid would share them:
        latitude  labels + ticks -> left column only
        longitude labels + ticks -> bottom row only
    Each panel still carries its own colorbar, because the three rows are
    on very different scales.

    Output names are what Poster/figs/ expects:
        <stem>_06mo.png   <stem>_01yr.png   <stem>_final.png
    """
    rows = rows or POSTER_ROWS
    n    = poster_frame_numbers(ds, stride=stride, leads_days=leads, quiet=True)

    # 1. prepare every row: mask, single level, one colour limit per row
    fields = [prepare_ADJ_field(r["var"], ds, zlev=r.get("zlev"),
                                p=p, ylim=ylim)
              for r in rows]

    # 2. one layout for all nine panels
    ax_rect, cb_rect = _common_layout(fields, style=style)
    print("shared layout:")
    print("    map  rect =", tuple(round(v, 4) for v in ax_rect))
    print("    cbar rect =", tuple(round(v, 4) for v in cb_rect))

    os.makedirs(outdir, exist_ok=True)
    idx = np.arange(0, fields[0][0].sizes["time"], stride)[::-1]
    last_row = len(rows) - 1

    for ri, (r, (da, xcoord, ycoord, vmax)) in enumerate(zip(rows, fields)):
        depth = (None if r.get("zlev") is None
                 else abs(float(ds.Z.values[r["zlev"]])))

        print(f"{r['var']}: scale = ±{vmax:.3e} (p{p})"
              + (f", depth = {depth:.0f} m" if depth is not None
                 else ", surface field"))

        for ci, (lead, suf) in enumerate(zip(leads, suffixes)):
            i = idx[n[lead]]

            try:
                tstr = np.datetime_as_string(da.time.values[i], unit="D")
            except Exception:
                tstr = str(da.time.values[i])

            title = (tstr if depth is None
                     else f"{tstr}   •   depth = {depth:.0f} m")

            fig, ax, cax = _make_panel(
                da, i, xcoord, ycoord, vmax, title,
                show_x=(ri == last_row),   # longitude on the bottom row
                show_y=(ci == 0),          # latitude on the left column
                rects=(ax_rect, cb_rect),
                style=style,
            )

            fname = os.path.join(outdir, f"{r['stem']}_{suf}.{filetype}")
            fig.savefig(fname, dpi=fig.dpi, facecolor="white")  # no bbox_inches
            plt.close(fig)
            print(f"    {os.path.basename(fname):<26s} {title}")

    return {r["var"]: f[3] for r, f in zip(rows, fields)}


# ----------------------------------------------------------------------
#  A whole strided sequence of one variable (for rebuilding an animation).
#  Not used by the poster; panels are auto-fitted individually here, which
#  is fine because every frame of one variable shares the same labels.
# ----------------------------------------------------------------------

def save_ADJ_sequence(
    var, ds, zlev=None, outdir=None, stride=4, p=96, ylim=None,
    filetype="png", reverse_time=True, add_colorbar=True,
    title_mode="date+depth", style=None,
):
    """Save every stride-th frame of one ADJ variable as <var>_0000..."""
    da, xcoord, ycoord, vmax = prepare_ADJ_field(
        var, ds, zlev=zlev, p=p, ylim=ylim
    )
    depth = None if zlev is None else abs(float(ds.Z.values[zlev]))

    idx = np.arange(0, da.sizes["time"], stride)
    if reverse_time:
        idx = idx[::-1]

    outdir = outdir or os.path.join(run_dir, "figures", f"{var}_frames")
    os.makedirs(outdir, exist_ok=True)

    st = {**POSTER_STYLE, **(style or {})}
    print(f"{var}: {len(idx)} frames -> {outdir}/   scale = ±{vmax:.3e} (p{p})")

    for nfr, i in enumerate(idx):
        try:
            tstr = np.datetime_as_string(da.time.values[i], unit="D")
        except Exception:
            tstr = str(da.time.values[i])

        if title_mode == "date":
            title = tstr
        elif title_mode == "date+depth":
            title = (tstr if depth is None
                     else f"{tstr}   •   depth = {depth:.0f} m")
        elif title_mode == "full":
            title = (f"{var} • time={tstr}" if depth is None
                     else f"{var} • depth={depth:.1f} m • time={tstr}")
        else:
            title = title_mode.format(var=var, time=tstr, depth=depth)

        fig, ax, cax = _make_panel(da, i, xcoord, ycoord, vmax, title,
                                   add_colorbar=add_colorbar, style=style)
        _fit_axes_in_figure(fig, [a for a in (ax, cax) if a is not None],
                            pad=st["pad"])
        fig.savefig(os.path.join(outdir, f"{var}_{nfr:04d}.{filetype}"),
                    dpi=fig.dpi, facecolor="white")
        plt.close(fig)

    print("    done.")
    return vmax

In [11]:
# ======================================================================
#  Generate the nine poster panels  ->  poster_frames/*.png
#
#  One call, so all nine share a layout and the columns line up.
#  Copy the contents of poster_frames/ into Poster/figs/ and recompile.
# ======================================================================

poster_frame_numbers(ds_adj, stride=4)      # sanity check the frame numbers

save_poster_grid(ds_adj, stride=4, p=96)

dt = 5 d, stride = 4 -> 20 d per frame; final frame = 91
       180 d -> frame    9
       365 d -> frame   18
       final -> frame   91
shared layout:
    map  rect = (0.0893, 0.135, 0.8044, 0.7949)
    cbar rect = (0.9178, 0.135, 0.0167, 0.7949)
ADJtheta: scale = ±2.791e-05 (p96), depth = 231 m
    sens_theta_06mo.png        2184-11-11   •   depth = 231 m
    sens_theta_01yr.png        2184-05-15   •   depth = 231 m
    sens_theta_final.png       2180-05-16   •   depth = 231 m
ADJdiffkr: scale = ±5.097e-02 (p96), depth = 231 m
    sens_diffkr_06mo.png       2184-11-11   •   depth = 231 m
    sens_diffkr_01yr.png       2184-05-15   •   depth = 231 m
    sens_diffkr_final.png      2180-05-16   •   depth = 231 m
ADJtaux: scale = ±3.286e-07 (p96), surface field
    sens_taux_06mo.png         2184-11-11
    sens_taux_01yr.png         2184-05-15
    sens_taux_final.png        2180-05-16


{'ADJtheta': 2.790789483697152e-05,
 'ADJdiffkr': 0.05096889942884439,
 'ADJtaux': 3.286489993570285e-07}

In [12]:
# ======================================================================
#  Optional variations
# ======================================================================

# 1. Crop the far south.  Below about 50S these fields are essentially zero
#    in every frame, so ~15% of each panel is dead white space.  ylim is
#    applied to all nine panels at once, so the rows stay comparable.
#
# save_poster_grid(ds_adj, p=96, ylim=(-55, 70), outdir="poster_frames_crop")


# 2. Stretch one row's colour scale.  The diffkr row reads faint because
#    p=96 is set by the strong western-boundary signal; a lower percentile
#    brings out the interior.  Only that row changes -- each row already
#    carries its own scale and its own colorbar.
#
# rows = [dict(POSTER_ROWS[0]), dict(POSTER_ROWS[1]), dict(POSTER_ROWS[2])]
# save_poster_grid(ds_adj, rows=rows, p=92, outdir="poster_frames_p92")


# 3. One colorbar per ROW instead of one per panel.  The scale is constant
#    along a row, so the first two columns do not really need one.  NOTE
#    this breaks the shared layout unless you keep the colorbar axes in
#    place and simply hide it, so it is not wired up here -- ask if you
#    want it.


# 4. Rebuild a full animation sequence with the poster styling, e.g. for
#    ADJtaux, which the old notebook never saved frames for.
#
# save_ADJ_sequence("ADJtaux", ds_adj, outdir="adj_taux_2d",
#                   stride=4, p=96, title_mode="full")


# 5. Tile-boundary check for the vertical stripes in ADJtaux.  The domain
#    is 51 points wide and 51 = 3 x 17, so a three-tile decomposition puts
#    tile edges at i = 17 and i = 34 -- exactly where the stripes sit.
#    If those two columns stand out here, they are a decomposition
#    artefact in the adjoint rather than signal.
#
# a    = np.abs(ds_adj.ADJtaux.isel(time=-10).values)
# prof = np.nanmean(a, axis=0)                 # mean |ADJtaux| per longitude
# print("loudest columns:", np.argsort(prof)[-6:])
# print("i=16,17,18 ->", prof[16:19])
# print("i=33,34,35 ->", prof[33:36])